# FL Attack Simulation

Simulates adversarial FL rounds under the attack types described in the TARS paper (Ahmed et al., 2025) and the GRADF framework:

| Attack | Surface | Description |
|--------|---------|-------------|
| `sign_flipping` | gradient | `w = -Delta` — pure sign reversal |
| `gaussian_noise` | gradient | `w = Delta + epsilon`, `epsilon ~ N(0, sigma^2 I)` |
| `poisoning` | gradient | Scale update by 10–50x |
| `sybil` | gradient | Amplified sign-flip simulating N fake identities |
| `colluding` | gradient | Coordinated sign-flip with boost |
| `free_riding` | gradient | Submit zero update |
| `label_flipping` | data | `y <- (y + 1) mod K` before local training |
| `pretense` | gradient | Honest for T rounds, then sign_flipping |

> **Setup:** N=10 clients, f=2 Byzantine (20%), MNIST non-IID — matching the TARS paper experimental setup.

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('src.fl.federated_learner').setLevel(logging.WARNING)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.fl.federated_learner import FederatedLearner, ParticipantData, RoundResult
from src.classification.attack_simulator import AttackSimulator

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)


class AttackedFederatedLearner(FederatedLearner):
    '''
    FederatedLearner extended with per-round attack injection.

    Byzantine clients (by index in byzantine_ids) have their parameter
    updates poisoned after local training. For label_flipping, training
    labels are corrupted before local training (data-level attack).

    pretense_rounds > 0: clients behave honestly for rounds 1..pretense_rounds
    then launch attacks from round pretense_rounds+1 (TARS pretense attack).
    '''

    def __init__(self, *args, attack_type='none', byzantine_ids=None,
                 pretense_rounds=0, n_classes_data=10, seed=42, **kwargs):
        super().__init__(*args, **kwargs)
        self.attack_type = attack_type
        self.byzantine_ids = set(byzantine_ids or [])
        self.pretense_rounds = pretense_rounds
        self.n_classes_data = n_classes_data
        self._sim = AttackSimulator(seed=seed)

    def _run_round(self, round_num, participants, root_data):
        active = (
            self.attack_type not in ('none', '') and
            round_num > self.pretense_rounds
        )

        param_updates = []
        for i, p in enumerate(participants):
            is_byz = (i in self.byzantine_ids) and active
            model = self._make_model(p.n_features)

            if is_byz and self.attack_type == 'label_flipping':
                _, y_p = self._sim.poison_data(
                    p.X_train, p.y_train, 'label_flipping',
                    n_classes=self.n_classes_data,
                )
                delta = model.fit(p.X_train, y_p)
            else:
                delta = model.fit(p.X_train, p.y_train)
                if is_byz:
                    delta = self._sim.poison_update(delta, self.attack_type)

            param_updates.append(delta)

        agg_kwargs = {}
        if self.aggregation == 'fltrust':
            srv = self._make_model(participants[0].n_features)
            srv_delta = srv.fit(root_data['X'], root_data['y'])
            agg_kwargs['server_update'] = srv_delta
        else:
            agg_kwargs['sample_sizes'] = [p.n_train for p in participants]

        agg_delta, metadata = self._strategy.aggregate(param_updates, **agg_kwargs)
        self._global_params = self._global_params + agg_delta

        per_acc = {
            p.id: self._make_model(p.n_features).accuracy(p.X_test, p.y_test)
            for p in participants
        }
        global_acc = float(np.mean(list(per_acc.values())))

        trust_scores, n_accepted = None, None
        if metadata and 'trust_scores' in metadata:
            raw = metadata['trust_scores']
            trust_scores = {p.id: ts for p, ts in zip(participants, raw)}
            n_accepted = sum(1 for ts in raw if ts > 0)

        return RoundResult(round_num, global_acc, per_acc, trust_scores, n_accepted)


print('AttackedFederatedLearner defined')

In [ ]:
DATA_ROOT = '../data/processed'
DATASET   = 'mnist'
SPLIT     = 'non_iid'
N_CLIENTS = 10
N_CLASSES = 10
N_ROUNDS  = 20
BYZANTINE_FRACTION = 0.2   # 20% Byzantine — TARS paper setup

N_BYZ   = max(1, int(N_CLIENTS * BYZANTINE_FRACTION))
BYZ_IDS = list(range(N_BYZ))   # first N_BYZ clients are Byzantine

participants = [
    ParticipantData.from_npy(f'{DATA_ROOT}/{DATASET}/{SPLIT}/client_{i}')
    for i in range(N_CLIENTS)
]
root_data = ParticipantData.load_server_val(
    f'{DATA_ROOT}/{DATASET}/{SPLIT}/server_val'
)

print(f'Dataset:   {DATASET.upper()} {SPLIT}')
print(f'Clients:   {N_CLIENTS} total, {N_BYZ} Byzantine ({BYZANTINE_FRACTION*100:.0f}%)')
print(f'Byzantine: client indices {BYZ_IDS}')
print(f'Features:  {participants[0].n_features}  (flattened 28x28)')
print(f'Rounds:    {N_ROUNDS}')

## 1. Baseline — no attack

Establishes the ceiling: both aggregation strategies with the same Byzantine clients behaving honestly.

In [ ]:
print('Baseline (no attack):')
baseline = {}
for agg in ['fltrust', 'fedavg']:
    print(f'  {agg} ...')
    learner = AttackedFederatedLearner(
        n_rounds=N_ROUNDS, aggregation=agg, n_classes=N_CLASSES,
        attack_type='none', byzantine_ids=BYZ_IDS,
    )
    h = learner.train(
        participants,
        root_data=root_data if agg == 'fltrust' else None,
        verbose=False,
    )
    baseline[agg] = [r.global_accuracy for r in h]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, N_ROUNDS+1), baseline['fltrust'], label='FLTrust',
        color='steelblue', marker='o', ms=3, lw=2)
ax.plot(range(1, N_ROUNDS+1), baseline['fedavg'],  label='FedAvg',
        color='darkorange', marker='s', ms=3, lw=2)
ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title(f'Baseline — no attack — {DATASET.upper()} {SPLIT}, {N_CLIENTS} clients')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/08_baseline.png', bbox_inches='tight')
plt.show()

for agg, accs in baseline.items():
    print(f'{agg.upper():10s}: final={accs[-1]:.4f}')

## 2. Gradient-level attacks under FedAvg

FedAvg has no Byzantine-robustness mechanism, so all attacks should degrade accuracy significantly.

In [ ]:
GRADIENT_ATTACKS = ['sign_flipping', 'gaussian_noise', 'poisoning',
                    'sybil', 'colluding', 'free_riding']

print(f'Gradient attacks under FedAvg ({N_BYZ}/{N_CLIENTS} Byzantine):')
results_fedavg = {'no_attack': baseline['fedavg']}

for atk in GRADIENT_ATTACKS:
    print(f'  {atk} ...')
    learner = AttackedFederatedLearner(
        n_rounds=N_ROUNDS, aggregation='fedavg', n_classes=N_CLASSES,
        attack_type=atk, byzantine_ids=BYZ_IDS,
    )
    h = learner.train(participants, root_data=None, verbose=False)
    results_fedavg[atk] = [r.global_accuracy for r in h]

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.tab10(np.linspace(0, 0.85, len(results_fedavg)))
for (name, accs), c in zip(results_fedavg.items(), cmap):
    kw = {'lw': 2.5, 'color': 'black'} if name == 'no_attack' else {'lw': 1.5, 'color': c, 'ls': '--'}
    ax.plot(range(1, N_ROUNDS+1), accs, label=name.replace('_', ' '), **kw)

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title(f'Attack impact on FedAvg — {DATASET.upper()} non-IID, {N_BYZ}/{N_CLIENTS} Byzantine')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/08_attacks_fedavg.png', bbox_inches='tight')
plt.show()

print('\nFinal accuracy under FedAvg:')
for name, accs in results_fedavg.items():
    drop = baseline['fedavg'][-1] - accs[-1]
    print(f'  {name:22s}: {accs[-1]:.4f}  (drop={drop:+.4f})')

## 3. FLTrust robustness vs FedAvg

FLTrust uses cosine-similarity trust scores to down-weight updates that diverge from the server reference gradient.

In [ ]:
print('FLTrust under same attacks:')
results_fltrust = {'no_attack': baseline['fltrust']}

for atk in GRADIENT_ATTACKS:
    print(f'  FLTrust + {atk} ...')
    learner = AttackedFederatedLearner(
        n_rounds=N_ROUNDS, aggregation='fltrust', n_classes=N_CLASSES,
        attack_type=atk, byzantine_ids=BYZ_IDS,
    )
    h = learner.train(participants, root_data=root_data, verbose=False)
    results_fltrust[atk] = [r.global_accuracy for r in h]

# Bar chart: final accuracy FedAvg vs FLTrust
attack_names = list(results_fedavg.keys())
fa_fedavg  = [results_fedavg[a][-1]  for a in attack_names]
fa_fltrust = [results_fltrust[a][-1] for a in attack_names]
x = np.arange(len(attack_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, fa_fedavg,  w, label='FedAvg',  color='darkorange', alpha=0.85)
ax.bar(x + w/2, fa_fltrust, w, label='FLTrust', color='steelblue',  alpha=0.85)
ax.set_xlabel('Attack type')
ax.set_ylabel(f'Final accuracy (round {N_ROUNDS})')
ax.set_title(f'FLTrust vs FedAvg robustness — {DATASET.upper()} non-IID, {N_BYZ}/{N_CLIENTS} Byzantine')
ax.set_xticks(x)
ax.set_xticklabels([a.replace('_', ' ') for a in attack_names], rotation=25, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/08_fltrust_vs_fedavg.png', bbox_inches='tight')
plt.show()

print(f'\n{"Attack":<25} {"FedAvg":>8} {"FLTrust":>8} {"Gain":>7}')
print('-' * 52)
for atk, fa, ft in zip(attack_names, fa_fedavg, fa_fltrust):
    print(f'{atk:<25} {fa:>8.4f} {ft:>8.4f} {ft-fa:>+7.4f}')

## 4. Byzantine fraction sweep

How many Byzantine clients can FLTrust tolerate before accuracy collapses?

In [ ]:
fractions = [0.1, 0.2, 0.3, 0.4, 0.5]
ATTACK_SWEEP = 'sign_flipping'

print(f'Byzantine fraction sweep — {ATTACK_SWEEP}, FLTrust vs FedAvg:')
byz_results = {'fltrust': {}, 'fedavg': {}}

for frac in fractions:
    n_byz_f   = max(1, int(N_CLIENTS * frac))
    byz_ids_f = list(range(n_byz_f))
    for agg in ['fltrust', 'fedavg']:
        print(f'  frac={frac:.1f}, {agg} ...')
        learner = AttackedFederatedLearner(
            n_rounds=N_ROUNDS, aggregation=agg, n_classes=N_CLASSES,
            attack_type=ATTACK_SWEEP, byzantine_ids=byz_ids_f,
        )
        h = learner.train(
            participants,
            root_data=root_data if agg == 'fltrust' else None,
            verbose=False,
        )
        byz_results[agg][frac] = [r.global_accuracy for r in h]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
cmap = plt.cm.Reds(np.linspace(0.3, 0.9, len(fractions)))
for ax, agg in zip(axes, ['fltrust', 'fedavg']):
    for frac, c in zip(fractions, cmap):
        accs = byz_results[agg][frac]
        ax.plot(range(1, N_ROUNDS+1), accs, label=f'{frac*100:.0f}% byz', color=c, lw=1.8)
    ax.set_title(agg.upper())
    ax.set_xlabel('Round')
    ax.set_ylabel('Global accuracy')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)
fig.suptitle(f'Byzantine fraction impact — {ATTACK_SWEEP.replace("_", " ")} attack')
plt.tight_layout()
plt.savefig('../results/figures/08_fraction_sweep.png', bbox_inches='tight')
plt.show()

print(f'\nFinal accuracy at round {N_ROUNDS}:')
print(f'{"Fraction":>10} {"FLTrust":>10} {"FedAvg":>10}')
for frac in fractions:
    ft = byz_results['fltrust'][frac][-1]
    fa = byz_results['fedavg'][frac][-1]
    print(f'{frac*100:>9.0f}% {ft:>10.4f} {fa:>10.4f}')

## 5. Pretense attack (TARS paper)

Byzantine clients behave honestly for T rounds to accumulate trust, then launch sign_flipping. This defeats detectors that rely on historical trust scores.

In [ ]:
PRETENSE_ROUNDS = 5
print(f'Pretense: honest for rounds 1-{PRETENSE_ROUNDS}, then sign_flipping')

pretense_results = {}
for agg in ['fltrust', 'fedavg']:
    print(f'  {agg} ...')
    learner = AttackedFederatedLearner(
        n_rounds=N_ROUNDS, aggregation=agg, n_classes=N_CLASSES,
        attack_type='sign_flipping', byzantine_ids=BYZ_IDS,
        pretense_rounds=PRETENSE_ROUNDS,
    )
    h = learner.train(
        participants,
        root_data=root_data if agg == 'fltrust' else None,
        verbose=False,
    )
    pretense_results[agg] = [r.global_accuracy for r in h]

fig, ax = plt.subplots(figsize=(10, 4))
ax.axvspan(0.5, PRETENSE_ROUNDS + 0.5,
           alpha=0.07, color='green', label='Pretense phase (honest)')
ax.axvspan(PRETENSE_ROUNDS + 0.5, N_ROUNDS + 0.5,
           alpha=0.07, color='red',   label='Attack phase (sign_flipping)')
ax.axvline(PRETENSE_ROUNDS + 0.5, color='gray', lw=1, ls='--')

ax.plot(range(1, N_ROUNDS+1), baseline['fltrust'],
        color='steelblue',  ls=':', lw=1.5, label='FLTrust baseline')
ax.plot(range(1, N_ROUNDS+1), baseline['fedavg'],
        color='darkorange', ls=':', lw=1.5, label='FedAvg baseline')
ax.plot(range(1, N_ROUNDS+1), pretense_results['fltrust'],
        color='steelblue',  ls='-', lw=2.2, label='FLTrust + pretense')
ax.plot(range(1, N_ROUNDS+1), pretense_results['fedavg'],
        color='darkorange', ls='-', lw=2.2, label='FedAvg + pretense')

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title(f'Pretense attack (T={PRETENSE_ROUNDS} honest -> sign_flipping)')
ax.legend(fontsize=9, loc='lower left')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/08_pretense_attack.png', bbox_inches='tight')
plt.show()

print('\nFinal accuracy after pretense attack:')
for agg in ['fltrust', 'fedavg']:
    pr = pretense_results[agg][-1]
    bl = baseline[agg][-1]
    print(f'  {agg.upper():10s}: {pr:.4f}  (baseline={bl:.4f}, drop={pr-bl:+.4f})')

## 6. Label flipping (data-level attack)

Byzantine clients corrupt their training labels before local training: `y <- (y + 1) mod K`. The submitted gradient looks plausible in magnitude, making it harder for FLTrust to detect via cosine similarity alone.

In [ ]:
print('Label flipping (data-level) — FLTrust vs FedAvg:')
lf_results = {}
for agg in ['fltrust', 'fedavg']:
    print(f'  {agg} ...')
    learner = AttackedFederatedLearner(
        n_rounds=N_ROUNDS, aggregation=agg, n_classes=N_CLASSES,
        attack_type='label_flipping', byzantine_ids=BYZ_IDS,
        n_classes_data=N_CLASSES,
    )
    h = learner.train(
        participants,
        root_data=root_data if agg == 'fltrust' else None,
        verbose=False,
    )
    lf_results[agg] = [r.global_accuracy for r in h]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, N_ROUNDS+1), baseline['fltrust'],
        color='steelblue',  ls=':', lw=1.5, label='FLTrust baseline')
ax.plot(range(1, N_ROUNDS+1), baseline['fedavg'],
        color='darkorange', ls=':', lw=1.5, label='FedAvg baseline')
ax.plot(range(1, N_ROUNDS+1), lf_results['fltrust'],
        color='steelblue',  ls='-', lw=2, label='FLTrust + label_flipping')
ax.plot(range(1, N_ROUNDS+1), lf_results['fedavg'],
        color='darkorange', ls='-', lw=2, label='FedAvg + label_flipping')

ax.set_xlabel('Round')
ax.set_ylabel('Global accuracy')
ax.set_title('Label flipping attack — FLTrust vs FedAvg')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/figures/08_label_flipping.png', bbox_inches='tight')
plt.show()

for agg in ['fltrust', 'fedavg']:
    lf = lf_results[agg][-1]
    bl = baseline[agg][-1]
    print(f'{agg.upper():10s}: {lf:.4f}  (baseline={bl:.4f}, drop={lf-bl:+.4f})')

## 7. Summary

Final accuracy for all tested attack/aggregation combinations.

In [ ]:
summary_fa, summary_ft = {}, {}

for atk, accs in results_fedavg.items():
    summary_fa[atk] = accs[-1]
for atk, accs in results_fltrust.items():
    summary_ft[atk] = accs[-1]

summary_fa['label_flipping']     = lf_results['fedavg'][-1]
summary_ft['label_flipping']     = lf_results['fltrust'][-1]
summary_fa['pretense_sign_flip'] = pretense_results['fedavg'][-1]
summary_ft['pretense_sign_flip'] = pretense_results['fltrust'][-1]

all_attack_keys = sorted(set(summary_fa) | set(summary_ft))
df = pd.DataFrame({
    'FedAvg':  [summary_fa.get(k, float('nan')) for k in all_attack_keys],
    'FLTrust': [summary_ft.get(k, float('nan')) for k in all_attack_keys],
}, index=all_attack_keys)
df.index.name = 'Attack'
df['FLTrust gain'] = df['FLTrust'] - df['FedAvg']
df = df.sort_values('FLTrust gain', ascending=False)

print(f'Final accuracy (round {N_ROUNDS}) — {DATASET.upper()} non-IID, {N_BYZ}/{N_CLIENTS} Byzantine')
print('=' * 52)
print(df.to_string(float_format='{:.4f}'.format))

df.to_csv('../results/tables/08_attack_simulation_summary.csv')
print('\nSaved to ../results/tables/08_attack_simulation_summary.csv')